In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-03 15:52:34.498751: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-03 15:52:35.269678: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": "local",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions


In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-03 15:52:36,958 [DEBUG] [Rain] Rain is initialized
2023-07-03 15:52:36,961 [DEBUG] [Provisioner] Creating coordinator
2023-07-03 15:52:36,962 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/coord/data/
2023-07-03 15:52:36,964 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-03 15:52:36,965 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-03 15:52:36,966 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/coord/data/
2023-07-03 15:52:36,967 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized
2023-07-03 15:52:36,968 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/
2023-07-03 15:52:36,969 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/
2023-07-03 15:52:36,970 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/data/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-03 15:52:36,977 [DEBUG] [Rain] Creating workers
2023-07-03 15:52:36,983 [INFO] [Provisioner] provisioner is serving
2023-07-03 15:52:36,984 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 15:52:36,985 [INFO] [Coordinator] coordinator is serving
2023-07-03 15:52:36,985 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 15:52:36,989 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 15:52:36,991 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 15:52:36,992 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 15:52:36,992 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 15:52:36,995 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 15:52:36,996 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 15:52:36,998

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 3s 12ms/step - loss: 0.7110 - accuracy: 0.7782
Epoch 2/2
157/157 [==============================] - 3s 12ms/step - loss: 0.6944 - accuracy: 0.7793
Epoch 2/2
157/157 [==============================] - 2s 11ms/step - loss: 0.3053 - accuracy: 0.9076
sending data to coordinator
sending data to coordinator


2023-07-03 15:53:18,569 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 15:53:18,570 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 15:53:18,587 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 15:53:18,588 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 15:53:18,594 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 15:53:18,595 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 15:53:18,798 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2 successfully
2023-07-03 15:53:18,816 [DEBUG] [DeepLearning] Asynchronous update is done by

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 11ms/step - loss: 0.2943 - accuracy: 0.9097
Epoch 2/2
157/157 [==============================] - 3s 12ms/step - loss: 0.3929 - accuracy: 0.8834
Epoch 2/2
157/157 [==============================] - 3s 11ms/step - loss: 0.2605 - accuracy: 0.9221
Epoch 2/2
157/157 [==============================] - 2s 11ms/step - loss: 0.2014 - accuracy: 0.9393
sending data to coordinator


2023-07-03 15:53:28,036 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 15:53:28,038 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 15:53:28,115 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1 successfully
2023-07-03 15:53:28,121 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-03 15:53:28,142 [DEBUG] [DeepLearning] Iteration 2/3 complete for worker 1.
2023-07-03 15:53:28,143 [DEBUG] [DeepLearning] Starting iteration 3/3
2023-07-03 15:53:28,143 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-03 15:53:28,144 [DEBUG] [DividerAmbassador] Sending ../../../../Rain/data/divider/data/1.pkl to worker1
2023-07-03 15:53:28,159 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 15:53:28,160 [DEBUG] [DividerAmbassador] divider 

...

2023-07-03 15:53:39,816 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3 successfully
2023-07-03 15:53:39,850 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3


 67/157 [===========>..................] - ETA: 1s - loss: 0.1472 - accuracy: 0.9571

2023-07-03 15:53:39,977 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.


157/157 [==============================] - 2s 12ms/step - loss: 0.1493 - accuracy: 0.9560
sending data to coordinator


2023-07-03 15:53:41,789 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 15:53:41,790 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 15:53:41,865 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2 successfully
2023-07-03 15:53:41,871 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
2023-07-03 15:53:41,891 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
2023-07-03 15:53:41,892 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-03 15:53:41,893 [DEBUG] [Divider] Divider stopped serving
2023-07-03 15:53:41,893 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-03 15:53:41,894 [DEBUG] [Divider] Divider stopped serving
2023-07-03 15:53:41,895 [INFO] [Provisioner] provisioner stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 7ms/step - loss: 0.1094 - accuracy: 0.9674

Test accuracy: 96.7%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-03 15:53:42,738 [DEBUG] [Rain] Creating workers
2023-07-03 15:53:42,745 [INFO] [Provisioner] provisioner is serving
2023-07-03 15:53:42,751 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 15:53:42,759 [INFO] [Coordinator] coordinator is serving
2023-07-03 15:53:42,763 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 15:53:42,767 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 15:53:42,769 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 15:53:42,775 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 15:53:42,781 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/data/
2023-07-03 15:53:42,783 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 15:53:42,783 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 15:53:42,785 [DEBUG] [TemporaryFilesManager] Creating

2023-07-03 15:53:42,878 [DEBUG] [Divider] Data partitioned
2023-07-03 15:53:42,879 [DEBUG] [DividerAmbassador] divider is sending data to the coordinator
2023-07-03 15:53:49,819 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 15:53:49,884 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 15:53:57,209 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 15:53:57,277 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 15:54:02,833 [DEBUG] [DividerAmbassador] divider is sending data to the provisioner
2023-07-03 15:54:02,896 [DEBUG] [DividerAmbassador] divider received: File received successfully from coordinator
2023-07-03 15:54:02,898 [DEBUG] [DividerProxy] Training Started
2023-07-03 15:54:02,899 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-03 15:54:02,900 [DEBUG] [Provisioner] Received '' from

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 3s 10ms/step - loss: 0.1532 - accuracy: 0.9550
Epoch 2/2
157/157 [==============================] - 3s 10ms/step - loss: 0.1460 - accuracy: 0.9566
Epoch 2/2
157/157 [==============================] - 2s 11ms/step - loss: 0.1318 - accuracy: 0.9610
sending data to coordinator


2023-07-03 15:54:31,700 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 15:54:31,701 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1
2023-07-03 15:54:31,821 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/1_1_trained.pkl from worker1 successfully
2023-07-03 15:54:31,899 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 15:54:31,899 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_1_trained.pkl from worker2
2023-07-03 15:54:31,991 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_1_trained.pkl from worker2 successfully


sending data to coordinator


2023-07-03 15:54:33,069 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 15:54:33,070 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_1_trained.pkl from worker3
2023-07-03 15:54:33,151 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_1_trained.pkl from worker3 successfully
2023-07-03 15:54:33,178 [DEBUG] [DeepLearning] Iteration 1/3 complete.
2023-07-03 15:54:33,179 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-03 15:54:33,205 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-03 15:54:33,205 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-03 15:54:33,206 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07-03 15:54:33,207 [DEBUG] [DividerAmbassador] Sending ../../../../Rain/data/divider/data/1.pkl to worker1
2023-07-03 15:54:33,208 [DEBUG] [DividerAmbassador] Sending ../../../../Rain/data/divider/data/2.pkl to worker2
2023-07-03 15:54:33,208 [DEBUG

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 9ms/step - loss: 0.1409 - accuracy: 0.9586
Epoch 2/2
157/157 [==============================] - 2s 9ms/step - loss: 0.1227 - accuracy: 0.9638
Epoch 2/2
141/157 [=========================>....] - ETA: 0s - loss: 0.1128 - accuracy: 0.9659sending data to coordinator
sending data to coordinator
157/157 [==============================] - 2s 10ms/step - loss: 0.1108 - accuracy: 0.9664
sending data to coordinator


2023-07-03 15:54:43,500 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 15:54:43,500 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_2_trained.pkl from worker3
2023-07-03 15:54:43,501 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 15:54:43,502 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_2_trained.pkl from worker1
2023-07-03 15:54:43,577 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 15:54:43,579 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_2_trained.pkl from worker2
2023-07-03 15:54:43,743 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/3_2_trained.pkl from worker3 successfully
2023-07-03 15:54:43,743 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 10ms/step - loss: 0.1094 - accuracy: 0.9665
Epoch 2/2
157/157 [==============================] - 1s 9ms/step - loss: 0.0982 - accuracy: 0.9693
sending data to coordinator
157/157 [==============================] - 2s 10ms/step - loss: 0.0988 - accuracy: 0.9694
sending data to coordinator
157/157 [==============================] - 2s 10ms/step - loss: 0.1074 - accuracy: 0.9657
sending data to coordinator


2023-07-03 15:54:53,832 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 15:54:53,833 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/2_3_trained.pkl from worker2
2023-07-03 15:54:53,935 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 15:54:53,937 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/1_3_trained.pkl from worker1
2023-07-03 15:54:53,956 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 15:54:53,958 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/data/3_3_trained.pkl from worker3
2023-07-03 15:54:54,030 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/data/2_3_trained.pkl from worker2 successfully
2023-07-03 15:54:54,141 [DEBUG] [DividerAmbassador] Downloaded ../../../../Ra

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 7ms/step - loss: 0.0763 - accuracy: 0.9772

Test accuracy: 97.7%
